In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_cohere import ChatCohere
from dotenv import load_dotenv
from typing import TypedDict

In [2]:
load_dotenv()
llm = ChatCohere(model="command-r7b-12-2024")

In [11]:
class BatsmanState(TypedDict):
    runs : int
    balls : int
    fours : int
    sixes : int

    sr : float
    bpb : float
    bp : float
    summary : str

def calculate_sr(state : BatsmanState):
    sr = (state['runs']/state['balls'])*100
    return {'sr':sr}

def calculate_bpb(state : BatsmanState):

    bpb = state['balls']/(state['fours']+state['sixes'])
    return {'bpb':bpb}

def calculate_bp(state : BatsmanState):

    bp = (state['fours']*4+state['sixes']*6)/state['runs']
    return {'bp':bp}

def summary(state : BatsmanState):
    summary = f"""
    strike rate - {state['sr']}
    balls per boundary - {state['bpb']}
    boundary percent - {state['bp']}
        """
    return {'summary':summary}

In [12]:
graph = StateGraph(BatsmanState)

graph.add_node('calculate_sr',calculate_sr)
graph.add_node('calculate_bpb',calculate_bpb)
graph.add_node('calculate_bp',calculate_bp)
graph.add_node('summary',summary)

graph.add_edge(START,'calculate_sr')
graph.add_edge(START,'calculate_bpb')
graph.add_edge(START,'calculate_bp')

graph.add_edge('calculate_sr','summary')
graph.add_edge('calculate_bpb','summary')
graph.add_edge('calculate_bp','summary')

graph.add_edge('summary',END)

workflow = graph.compile()

In [13]:
initial_state = {
    'balls':50,
    'runs':100,
    'fours':6,
    'sixes':4
}

final_state = workflow.invoke(initial_state)
print(final_state)

{'runs': 100, 'balls': 50, 'fours': 6, 'sixes': 4, 'sr': 200.0, 'bpb': 5.0, 'bp': 0.48, 'summary': '\n    strike rate - 200.0\n    balls per boundary - 5.0\n    boundary percent - 0.48\n        '}
